# Homework 5: Thuật toán Q-Learning
## Môn học: Trí Tuệ Nhân Tạo - EE3063

Nhóm Thực Hiện: [Điền tên thành viên nhóm]
MSSV: [Điền MSSV thành viên nhóm]

Nội dung:
1. Giới thiệu về Q-Learning và bài toán ví dụ (điều hướng trong các phòng).
2. Xây dựng thuật toán Q-Learning từ đầu.
3. Huấn luyện mô hình và hiển thị Q-table đã học.
4. Tìm đường đi tối ưu dựa trên Q-table.

In [1]:
# -*- coding: utf-8 -*-
import numpy as np
import random # Để chọn hành động ngẫu nhiên và trạng thái bắt đầu

## 1. Giới thiệu về Học Củng Cố và Q-Learning

**Học Củng Cố (Reinforcement Learning - RL)** là một lĩnh vực của học máy, nơi một **tác nhân (agent)** học cách tương tác với một **môi trường (environment)** để đạt được một mục tiêu cụ thể. Tác nhân thực hiện các **hành động (actions)** trong môi trường, và dựa trên hành động đó, môi trường sẽ chuyển sang một **trạng thái (state)** mới và trả về một tín hiệu **phần thưởng (reward)** (hoặc phạt). Mục tiêu của tác nhân là học một **chính sách (policy)** - tức là một chiến lược chọn hành động - sao cho tổng phần thưởng nhận được về lâu dài là lớn nhất.

**Q-Learning** là một thuật toán học củng cố không phụ thuộc mô hình (model-free), off-policy. Nó học một hàm giá trị hành động (action-value function), gọi là **Q-function** hoặc **Q-value**, $Q(s, a)$. Giá trị $Q(s, a)$ biểu thị "chất lượng" của việc thực hiện hành động $a$ tại trạng thái $s$, tức là tổng phần thưởng kỳ vọng trong tương lai nếu bắt đầu từ $s$, thực hiện $a$, và sau đó tuân theo chính sách tối ưu.

### Bảng Q (Q-Table)
Trong trường hợp không gian trạng thái và không gian hành động là rời rạc và hữu hạn, Q-function có thể được biểu diễn bằng một bảng gọi là Q-table. Các hàng của bảng tương ứng với các trạng thái, và các cột tương ứng với các hành động. Mỗi ô $Q[state, action]$ lưu trữ giá trị Q ước tính.

### Công thức cập nhật Q-value
Q-Learning cập nhật các giá trị Q một cách lặp đi lặp lại dựa trên kinh nghiệm mà tác nhân thu thập được. Công thức cập nhật Bellman cho Q-Learning là:

$Q(s, a) \leftarrow (1 - \alpha) \cdot Q(s, a) + \alpha \cdot [r + \gamma \cdot \max_{a'} Q(s', a')]$

Trong đó:
-   $s$: Trạng thái hiện tại.
-   $a$: Hành động hiện tại.
-   $r$: Phần thưởng nhận được sau khi thực hiện hành động $a$ tại trạng thái $s$.
-   $s'$: Trạng thái tiếp theo sau khi thực hiện hành động $a$.
-   $a'$: Tất cả các hành động có thể thực hiện tại trạng thái $s'$.
-   $\alpha$ (alpha): Tốc độ học (learning rate), $0 < \alpha \le 1$. Quyết định mức độ các giá trị Q mới ghi đè lên các giá trị cũ.
-   $\gamma$ (gamma): Hệ số chiết khấu (discount factor), $0 \le \gamma < 1$. Quyết định tầm quan trọng của các phần thưởng trong tương lai. Giá trị $\gamma$ gần 0 sẽ làm cho tác nhân tập trung vào phần thưởng trước mắt, trong khi giá trị gần 1 sẽ làm cho tác nhân cân nhắc các phần thưởng dài hạn.

(Tham khảo slide: "Q learning.pdf", đặc biệt là trang 2, 12, 16, 17)

## 2. Bài toán Ví dụ: Điều hướng trong các phòng

Chúng ta sẽ áp dụng Q-Learning cho bài toán điều hướng robot trong một môi trường gồm 6 phòng (đánh số từ 0 đến 5) như trong slide "Q learning.pdf" (trang 7-11).

-   **Trạng thái (States):** Vị trí hiện tại của robot (phòng 0, 1, 2, 3, 4, hoặc 5).
-   **Hành động (Actions):** Di chuyển từ phòng hiện tại sang một phòng liền kề.
-   **Phần thưởng (Rewards):**
    * 0 nếu di chuyển đến một phòng hợp lệ (không phải phòng đích).
    * 100 nếu di chuyển đến phòng đích (phòng 5).
    * -1 (hoặc một giá trị không xác định, không cho phép di chuyển) cho các hành động không thể thực hiện (ví dụ: di chuyển giữa hai phòng không có cửa nối).
-   **Mục tiêu:** Tìm đường đi ngắn nhất (hoặc tối ưu về phần thưởng) từ một phòng bất kỳ đến phòng 5.

### Ma trận Phần thưởng R (R-matrix)
Dựa trên slide (trang 8), ma trận R (Rewards) biểu diễn phần thưởng trực tiếp khi chuyển từ `state` (hàng) sang `next_state` (cột - cũng là hành động). Giá trị -1 thể hiện không có đường đi trực tiếp.

```
      Hành động (Đi đến phòng)
      0   1   2   3   4    5
TT 0 [-1, -1, -1, -1,  0,  -1]
TT 1 [-1, -1, -1,  0, -1, 100]
TT 2 [-1, -1, -1,  0, -1,  -1]
TT 3 [-1,  0,  0, -1,  0,  -1]
TT 4 [ 0, -1, -1,  0, -1, 100]
TT 5 [-1,  0, -1, -1,  0, 100]  (Phòng 5 là đích, các hành động từ phòng 5 đến phòng 5 hoặc phòng khác có thưởng 100/0)
```
Lưu ý: Trong slide, phòng 5 có thể đi đến phòng 1, 4, 5. Nếu đến 5 từ 5 (ở yên) thì thưởng 100.
"""

--- Định nghĩa Môi trường ---
Ma trận R (Rewards)
Hàng: trạng thái hiện tại, Cột: hành động (đi đến trạng thái tiếp theo)
Giá trị -1 nghĩa là không thể thực hiện hành động đó (không có cửa nối)

In [3]:
R = np.array([
    [-1, -1, -1, -1,  0, -1],  # Từ phòng 0 có thể đến phòng 4
    [-1, -1, -1,  0, -1, 100], # Từ phòng 1 có thể đến phòng 3 và phòng 5 (đích)
    [-1, -1, -1,  0, -1, -1],  # Từ phòng 2 có thể đến phòng 3
    [-1,  0,  0, -1,  0, -1],  # Từ phòng 3 có thể đến phòng 1, 2, 4
    [ 0, -1, -1,  0, -1, 100], # Từ phòng 4 có thể đến phòng 0, 3, 5 (đích)
    [-1,  0, -1, -1,  0, 100]  # Từ phòng 5 (đích) có thể đến phòng 1, 4, 5
], dtype=float)

# Khởi tạo Q-table với tất cả giá trị bằng 0
# Kích thước: (số_trạng_thái, số_hành_động)
# Ở đây, số hành động bằng số trạng thái (đi đến phòng X)
n_states = R.shape[0]
n_actions = R.shape[1]
Q = np.zeros((n_states, n_actions))

# --- Xây dựng Thuật toán Q-Learning ---
class QLearningAgent:
    def __init__(self, R_matrix, learning_rate=0.8, discount_factor=0.8, n_episodes=1000):
        """
        Khởi tạo tác nhân Q-Learning.
        Args:
            R_matrix (np.array): Ma trận phần thưởng.
            learning_rate (float): Alpha - Tốc độ học.
            discount_factor (float): Gamma - Hệ số chiết khấu.
            n_episodes (int): Số lượng episodes để huấn luyện.
        """
        self.R = R_matrix
        self.alpha = learning_rate
        self.gamma = discount_factor
        self.n_episodes = n_episodes
        self.n_states = R_matrix.shape[0]
        self.n_actions = R_matrix.shape[1]
        self.Q_table = np.zeros((self.n_states, self.n_actions))
        self.goal_state = 5 # Trạng thái đích là phòng 5

    def get_possible_actions(self, state):
        """
        Lấy danh sách các hành động có thể thực hiện từ một trạng thái.
        Hành động hợp lệ là khi R[state, action] != -1.
        """
        return np.where(self.R[state, :] != -1)[0]

    def train(self):
        """
        Huấn luyện Q-table.
        """
        print("Bắt đầu quá trình huấn luyện Q-Learning...")
        for episode in range(self.n_episodes):
            # Chọn một trạng thái bắt đầu ngẫu nhiên (không phải là trạng thái đích)
            current_state = random.choice([s for s in range(self.n_states) if s != self.goal_state])
            # Hoặc có thể bắt đầu từ tất cả các trạng thái không phải đích
            # current_state = random.choice(np.where(np.diag(self.R) != 100)[0])


            # Lặp cho đến khi đạt trạng thái đích
            # Để tránh vòng lặp vô hạn nếu Q-table chưa tốt, có thể giới hạn số bước trong 1 episode
            max_steps_per_episode = self.n_states * 2 # Ví dụ
            for step in range(max_steps_per_episode):
                # Chọn một hành động ngẫu nhiên từ các hành động có thể thực hiện
                possible_actions = self.get_possible_actions(current_state)
                if len(possible_actions) == 0:
                    # Không có hành động nào từ trạng thái này, hiếm khi xảy ra với R-matrix đã cho
                    break 
                action = random.choice(possible_actions)
                
                # Trạng thái tiếp theo chính là hành động được chọn (đi đến phòng đó)
                next_state = action
                
                # Tính giá trị Q tối đa cho trạng thái tiếp theo
                # Q(s', a') là các giá trị trong hàng next_state của Q-table
                # Chỉ xem xét các hành động có thể từ next_state
                possible_next_actions = self.get_possible_actions(next_state)
                if len(possible_next_actions) == 0:
                    max_q_next_state = 0 # Nếu không có hành động nào từ next_state
                else:
                    max_q_next_state = np.max(self.Q_table[next_state, possible_next_actions])
                
                # Cập nhật Q-value
                # Q(s, a) = (1-alpha)Q(s,a) + alpha * (R(s,a) + gamma * max_Q(s',a'))
                # Slide trang 17 dùng công thức này.
                # Slide trang 12 (ví dụ) dùng Q(s,a) = R(s,a) + gamma * max_Q(s',a'), tương đương alpha=1.
                # Chúng ta sẽ dùng công thức tổng quát với alpha.
                
                reward = self.R[current_state, action]
                
                self.Q_table[current_state, action] = \
                    (1 - self.alpha) * self.Q_table[current_state, action] + \
                    self.alpha * (reward + self.gamma * max_q_next_state)
                
                current_state = next_state
                if current_state == self.goal_state:
                    break # Đã đến đích, kết thúc episode
            
            if (episode + 1) % 100 == 0:
                print(f"Hoàn thành episode: {episode + 1}/{self.n_episodes}")
        
        print("Hoàn tất huấn luyện Q-Learning.")
        # Chuẩn hóa Q-table (tùy chọn, để dễ nhìn hơn bằng cách chia cho giá trị lớn nhất)
        # print("\nQ-table sau khi huấn luyện (trước khi chuẩn hóa):")
        # print(self.Q_table.astype(int))
        # if np.max(self.Q_table) > 0:
        #     self.Q_table = (self.Q_table / np.max(self.Q_table) * 100).astype(int)


    def get_optimal_path(self, start_state):
        """
        Tìm đường đi tối ưu từ start_state đến goal_state dựa trên Q-table đã học.
        Args:
            start_state (int): Trạng thái bắt đầu.
        Returns:
            list: Danh sách các trạng thái trên đường đi tối ưu.
                  Trả về None nếu không tìm thấy đường đi hoặc start_state là goal_state.
        """
        if start_state == self.goal_state:
            return [start_state]
        
        path = [start_state]
        current_state = start_state
        max_path_length = self.n_states * 2 # Giới hạn độ dài đường đi để tránh vòng lặp vô hạn

        for _ in range(max_path_length):
            # Chọn hành động có Q-value cao nhất từ trạng thái hiện tại
            # Chỉ xem xét các hành động có thể (R[current_state, action] != -1)
            possible_actions_from_current = self.get_possible_actions(current_state)
            if len(possible_actions_from_current) == 0:
                print(f"Không có hành động nào từ trạng thái {current_state}.")
                return None # Không có đường đi

            # Lấy Q-values cho các hành động có thể
            q_values_for_possible_actions = self.Q_table[current_state, possible_actions_from_current]
            
            # Tìm hành động tốt nhất
            best_action_index_in_possible = np.argmax(q_values_for_possible_actions)
            next_state = possible_actions_from_current[best_action_index_in_possible]
            
            path.append(next_state)
            current_state = next_state
            
            if current_state == self.goal_state:
                return path # Đã đến đích
        
        print(f"Không tìm thấy đường đến đích từ trạng thái {start_state} trong giới hạn bước.")
        return None # Không tìm thấy đường đi

## 3. Huấn luyện và Kết quả

### 3.1. Huấn luyện mô hình Q-Learning

Khởi tạo và huấn luyện tác nhân

Các tham số alpha và gamma có thể được điều chỉnh để xem ảnh hưởng

Slide ví dụ dùng gamma = 0.8 và dường như alpha = 1 (cập nhật trực tiếp)

Ta sẽ dùng alpha = 0.8, gamma = 0.8, n_episodes = 1000 (hoặc nhiều hơn để hội tụ tốt hơn)

In [4]:
q_agent = QLearningAgent(R_matrix=R, learning_rate=0.8, discount_factor=0.8, n_episodes=2000)
q_agent.train()

Bắt đầu quá trình huấn luyện Q-Learning...
Hoàn thành episode: 100/2000
Hoàn thành episode: 200/2000
Hoàn thành episode: 300/2000
Hoàn thành episode: 400/2000
Hoàn thành episode: 500/2000
Hoàn thành episode: 600/2000
Hoàn thành episode: 700/2000
Hoàn thành episode: 800/2000
Hoàn thành episode: 900/2000
Hoàn thành episode: 1000/2000
Hoàn thành episode: 1100/2000
Hoàn thành episode: 1200/2000
Hoàn thành episode: 1300/2000
Hoàn thành episode: 1400/2000
Hoàn thành episode: 1500/2000
Hoàn thành episode: 1600/2000
Hoàn thành episode: 1700/2000
Hoàn thành episode: 1800/2000
Hoàn thành episode: 1900/2000
Hoàn thành episode: 2000/2000
Hoàn tất huấn luyện Q-Learning.


### 3.2. Hiển thị Q-Table

Q-table sau khi huấn luyện sẽ chứa các giá trị Q ước tính cho mỗi cặp (trạng thái, hành động).
Giá trị Q cao hơn cho thấy hành động đó tốt hơn khi ở trạng thái tương ứng.

In [5]:
print("\n--- Q-Table sau khi huấn luyện ---")
# In Q-table với định dạng dễ đọc hơn
q_table_display = q_agent.Q_table.astype(int) # Làm tròn thành số nguyên để dễ nhìn
print("     Hành động (Đi đến phòng)")
print("Trạng thái |  0   1   2   3   4    5")
print("------------------------------------")
for i in range(q_agent.n_states):
    row_str = f"    {i}    | "
    for val in q_table_display[i]:
        row_str += f"{val:3d} "
    print(row_str)


--- Q-Table sau khi huấn luyện ---
     Hành động (Đi đến phòng)
Trạng thái |  0   1   2   3   4    5
------------------------------------
    0    |   0   0   0   0  80   0 
    1    |   0   0   0  64   0 100 
    2    |   0   0   0  64   0   0 
    3    |   0  80  51   0  80   0 
    4    |  64   0   0  64   0 100 
    5    |   0   0   0   0   0   0 


### 3.3. Tìm đường đi tối ưu

Sử dụng Q-table đã học, chúng ta có thể tìm đường đi tối ưu từ một phòng bắt đầu đến phòng đích (phòng 5).

In [6]:
print("\n--- Tìm đường đi tối ưu đến phòng 5 ---")
start_rooms = [0, 1, 2, 3, 4] # Các phòng có thể bắt đầu
for room in start_rooms:
    optimal_path = q_agent.get_optimal_path(room)
    if optimal_path:
        path_str = " -> ".join(map(str, optimal_path))
        print(f"Đường đi tối ưu từ phòng {room} đến phòng 5: {path_str}")
    else:
        print(f"Không tìm thấy đường đi từ phòng {room} đến phòng 5.")


--- Tìm đường đi tối ưu đến phòng 5 ---
Đường đi tối ưu từ phòng 0 đến phòng 5: 0 -> 4 -> 5
Đường đi tối ưu từ phòng 1 đến phòng 5: 1 -> 5
Đường đi tối ưu từ phòng 2 đến phòng 5: 2 -> 3 -> 1 -> 5
Đường đi tối ưu từ phòng 3 đến phòng 5: 3 -> 1 -> 5
Đường đi tối ưu từ phòng 4 đến phòng 5: 4 -> 5


## 4. Nhận xét và Kết luận

-   Thuật toán Q-Learning đã được xây dựng từ đầu và áp dụng cho bài toán điều hướng trong môi trường các phòng.
-   Q-table được cập nhật qua nhiều episodes, học được "giá trị" của việc thực hiện một hành động cụ thể tại một trạng thái nhất định.
-   Dựa trên Q-table đã học, tác nhân có thể tìm ra một chính sách (đường đi) tối ưu để đạt được mục tiêu (đến phòng 5).
-   Các tham số như `learning_rate` ($\alpha$), `discount_factor` ($\gamma$), và `n_episodes` có ảnh hưởng đến quá trình hội tụ và kết quả của Q-table.
    * $\alpha$ cao giúp học nhanh hơn nhưng có thể không ổn định.
    * $\gamma$ cao hơn khiến tác nhân quan tâm nhiều hơn đến phần thưởng dài hạn.
    * Số episodes nhiều hơn thường giúp Q-table hội tụ tốt hơn.
-   Trong ví dụ này, các hành động được chọn ngẫu nhiên trong quá trình huấn luyện (exploration). Trong các ứng dụng phức tạp hơn, có thể sử dụng các chiến lược exploration/exploitation (ví dụ: epsilon-greedy) để cân bằng giữa việc thử các hành động mới và khai thác các hành động đã biết là tốt.

-   **So sánh với slide:**
    * Ma trận R được định nghĩa tương tự như trong slide.
    * Q-table kết quả có thể khác một chút so với slide (trang 11) do sự ngẫu nhiên trong việc chọn trạng thái bắt đầu, chọn hành động, và số lượng episodes. Tuy nhiên, các giá trị Q cao nên tập trung vào các hành động dẫn đến phòng đích.
    * Đường đi tối ưu tìm được nên tương ứng với các đường có giá trị Q cao trong Q-table. Ví dụ, từ phòng 2, đường đi có thể là 2 -> 3 -> 1 -> 5 hoặc 2 -> 3 -> 4 -> 5.